# Fine-Tune English-to-Shona Translation In Colab

This notebook trains a local English-to-Shona translator that can sit in front of the Shona XTTS voice model.

The target pipeline is simple:

`English text -> Shona translator -> Shona XTTS voice`


## Runtime Notes

Use a GPU runtime in Colab. A T4 is enough for this notebook if the batch size is kept modest.

The notebook uses free datasets and a free base model:

- Dataset: `michsethowusu/english-shona_sentence-pairs_mt560`
- Base model: `Helsinki-NLP/opus-mt-en-sn`
- Evaluation set: FLORES-200 `eng_Latn` and `sna_Latn`


In [ ]:
# Step 1: check the GPU runtime.
!nvidia-smi || true

import torch

HAS_CUDA = torch.cuda.is_available()
print('cuda:', HAS_CUDA)
if HAS_CUDA:
    print('device:', torch.cuda.get_device_name(0))


In [ ]:
# Step 2: mount Drive so checkpoints survive Colab disconnects.
from google.colab import drive

drive.mount('/content/drive')

from pathlib import Path

WORK_DIR = Path('/content/en-sn-translation')
DRIVE_DIR = Path('/content/drive/MyDrive/shona-translation')
MODEL_DIR = DRIVE_DIR / 'opus-mt-en-sn-finetuned'
DATA_DIR = DRIVE_DIR / 'data'

WORK_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

print('work dir:', WORK_DIR)
print('drive dir:', DRIVE_DIR)


In [ ]:
# Step 3: install training dependencies.
!pip install -U pip setuptools wheel
!pip install -q 'transformers<5' datasets evaluate sacrebleu sentencepiece sacremoses accelerate


In [ ]:
# Step 4: load the English-Shona parallel dataset.
from datasets import DatasetDict, load_dataset

DATASET_NAME = 'michsethowusu/english-shona_sentence-pairs_mt560'
raw = load_dataset(DATASET_NAME)

print(raw)
split_name = 'train' if 'train' in raw else list(raw.keys())[0]
print('using split:', split_name)
print('columns:', raw[split_name].column_names)
print(raw[split_name][0])


In [ ]:
# Step 5: normalize the dataset into {en, sn} rows.
import re

def find_parallel_columns(dataset):
    cols = dataset.column_names
    lower = {c.lower(): c for c in cols}

    en_candidates = ['english', 'en', 'source', 'src', 'eng']
    sn_candidates = ['shona', 'sn', 'sna', 'target', 'tgt']

    en_col = next((lower[c] for c in en_candidates if c in lower), None)
    sn_col = next((lower[c] for c in sn_candidates if c in lower), None)
    if en_col and sn_col:
        return en_col, sn_col

    if 'translation' in cols:
        sample = dataset[0]['translation']
        keys = {k.lower(): k for k in sample.keys()}
        en_key = next((keys[c] for c in en_candidates if c in keys), None)
        sn_key = next((keys[c] for c in sn_candidates if c in keys), None)
        if en_key and sn_key:
            return ('translation', en_key), ('translation', sn_key)

    raise ValueError(f'Could not identify English/Shona columns from: {cols}')

def value_from(row, ref):
    if isinstance(ref, tuple):
        return row[ref[0]][ref[1]]
    return row[ref]

en_ref, sn_ref = find_parallel_columns(raw[split_name])
print('english column:', en_ref)
print('shona column:', sn_ref)

SPACE_RE = re.compile(r'\s+')

def clean_text(text):
    text = str(text or '').replace('\u00a0', ' ')
    return SPACE_RE.sub(' ', text).strip()

def normalize_batch(batch):
    out_en = []
    out_sn = []
    rows = [dict(zip(batch.keys(), values)) for values in zip(*batch.values())]
    for row in rows:
        en = clean_text(value_from(row, en_ref))
        sn = clean_text(value_from(row, sn_ref))
        if not en or not sn:
            continue
        if en.lower() == sn.lower():
            continue
        if len(en) < 3 or len(sn) < 3:
            continue
        if len(en) > 600 or len(sn) > 600:
            continue
        out_en.append(en)
        out_sn.append(sn)
    return {'en': out_en, 'sn': out_sn}

cleaned = raw[split_name].map(
    normalize_batch,
    batched=True,
    remove_columns=raw[split_name].column_names,
    desc='cleaning pairs',
)

cleaned = cleaned.to_pandas().drop_duplicates().reset_index(drop=True)
print('clean rows:', len(cleaned))
print(cleaned.head())


In [ ]:
# Step 6: create train/validation splits and keep a local copy in Drive.
from datasets import Dataset

MAX_ROWS = 120_000
MAX_VALIDATION_ROWS = 2_000
SEED = 42

if len(cleaned) > MAX_ROWS:
    cleaned = cleaned.sample(MAX_ROWS, random_state=SEED).reset_index(drop=True)

dataset = Dataset.from_pandas(cleaned, preserve_index=False)
splits = dataset.train_test_split(test_size=0.02, seed=SEED)
validation = splits['test']
if len(validation) > MAX_VALIDATION_ROWS:
    validation = validation.shuffle(seed=SEED).select(range(MAX_VALIDATION_ROWS))
dataset = DatasetDict({'train': splits['train'], 'validation': validation})

dataset.save_to_disk(str(DATA_DIR / 'english_shona_mt560_clean'))
print(dataset)
print(dataset['train'][0])


In [ ]:
# Step 7: load the base translator.
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

BASE_MODEL = 'Helsinki-NLP/opus-mt-en-sn'
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)

if HAS_CUDA:
    model = model.cuda()

print('loaded:', BASE_MODEL)


In [ ]:
# Step 8: tokenize the sentence pairs.
MAX_SOURCE_LENGTH = 160
MAX_TARGET_LENGTH = 160

def preprocess(batch):
    model_inputs = tokenizer(
        batch['en'],
        max_length=MAX_SOURCE_LENGTH,
        truncation=True,
    )
    labels = tokenizer(
        text_target=batch['sn'],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

tokenized = dataset.map(
    preprocess,
    batched=True,
    remove_columns=dataset['train'].column_names,
    desc='tokenizing',
)

print(tokenized)


In [ ]:
# Step 9: train. Reduce batch size if the GPU runs out of memory.
import inspect
import numpy as np
import evaluate
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments

sacrebleu = evaluate.load('sacrebleu')

def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [[label.strip()] for label in labels]
    return preds, labels

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)
    result = sacrebleu.compute(predictions=decoded_preds, references=decoded_labels)
    return {'bleu': result['score']}

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

args_kwargs = dict(
    output_dir=str(MODEL_DIR / 'runs'),
    overwrite_output_dir=True,
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=2,
    predict_with_generate=True,
    generation_max_length=160,
    fp16=HAS_CUDA,
    logging_steps=100,
    eval_steps=1000,
    save_steps=1000,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='bleu',
    greater_is_better=True,
    report_to='none',
)

arg_names = set(inspect.signature(Seq2SeqTrainingArguments).parameters)
if 'eval_strategy' in arg_names:
    args_kwargs['eval_strategy'] = 'steps'
else:
    args_kwargs['evaluation_strategy'] = 'steps'

training_args = Seq2SeqTrainingArguments(**args_kwargs)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['validation'],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()


In [ ]:
# Step 10: save the fine-tuned translator to Drive.
FINAL_MODEL_DIR = MODEL_DIR / 'final'
trainer.save_model(str(FINAL_MODEL_DIR))
tokenizer.save_pretrained(str(FINAL_MODEL_DIR))

print('saved model:', FINAL_MODEL_DIR)


In [ ]:
# Step 11: quick manual checks.
import torch

def translate(text, max_new_tokens=160):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=MAX_SOURCE_LENGTH)
    if HAS_CUDA:
        inputs = {k: v.cuda() for k, v in inputs.items()}
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, num_beams=4)
    return tokenizer.decode(output[0], skip_special_tokens=True)

samples = [
    'Good morning, how are you today?',
    'Send money to your family using EcoCash.',
    'The promotion gives customers a chance to win prizes every week.',
]

for text in samples:
    print('EN:', text)
    print('SN:', translate(text))
    print()


In [ ]:
# Step 12: optional FLORES-200 evaluation sample.
# FLORES is small and useful for sanity checks, but it is not the main training set.
from datasets import load_dataset

try:
    flores = load_dataset('facebook/flores', 'eng_Latn-sna_Latn', split='devtest')
    print(flores)
    print(flores[0])
except Exception as exc:
    print('FLORES access can require accepting terms on Hugging Face first.')
    print(type(exc).__name__, exc)


In [ ]:
# Step 13: zip the final model for download or copying back into WSL.
%cd /content/drive/MyDrive/shona-translation/opus-mt-en-sn-finetuned
!zip -r opus-mt-en-sn-finetuned-final.zip final
print('archive:', MODEL_DIR / 'opus-mt-en-sn-finetuned-final.zip')
